# EcoMove Mobility - Data Visualisation (Google Colab)
Run each cell in order (Shift+Enter). You will be asked to upload `EcoMoveMobility.xlsx` in Step 2.

## Step 1 - Get the project code

In [ ]:
!git clone -q https://github.com/JonathanAkyeampong/Data-Visualisation-Project.git
%cd Data-Visualisation-Project
!pip install -q dash openpyxl

## Step 2 - Upload the dataset
Click **Choose Files** and select your EcoMoveMobility.xlsx file.

In [ ]:
import os, shutil
from google.colab import files

os.makedirs('data', exist_ok=True)
uploaded = files.upload()
name = next(iter(uploaded))
shutil.move(name, 'data/EcoMoveMobility.xlsx')
print('Saved', name, '-> data/EcoMoveMobility.xlsx')

## Step 3 - Load and check the data

In [ ]:
import pandas as pd

raw = pd.read_excel('data/EcoMoveMobility.xlsx')
print(raw.shape)
raw.head()

In [ ]:
raw.info()
print('\nMissing values:', raw.isna().sum().sum())
print('Duplicate rows:', raw.duplicated().sum())

## Step 4 - Clean the data and derive measures

In [ ]:
from ecomove_analysis import load_data, data_quality_report

df = load_data()
data_quality_report(df)

## Step 5 - Summary by transport mode

In [ ]:
mode = df.groupby('Transport_Mode').agg(
    Passengers=('Passengers', 'sum'), Revenue=('Ticket_Revenue_EUR', 'sum'),
    Cost=('Operating_Cost_EUR', 'sum'), Avg_delay=('Average_Delay_Minutes', 'mean'),
    On_time=('On_Time_Percentage', 'mean'), Rating=('Customer_Rating', 'mean'))
mode['Cost_recovery'] = mode['Revenue'] / mode['Cost']
mode.sort_values('Passengers', ascending=False).round(2)

## Step 6 - Create and show the eight visualisations

In [ ]:
import ecomove_analysis as ea
from IPython.display import Image, display

for f in (ea.fig1_time_series, ea.fig2_city_demand, ea.fig3_correlation,
          ea.fig4_delay_distribution, ea.fig5_cost_recovery,
          ea.fig6_accessibility_heatmap, ea.fig7_delay_vs_rating, ea.fig8_event_days):
    f(df)

for png in sorted(os.listdir('figures')):
    if png.startswith('fig'):
        display(Image(f'figures/{png}', width=900))

## Step 7 - Interactive dashboard
The dashboard appears below this cell. Use the filters at the top to explore by date, city, mode, route type, weather and event day.

In [ ]:
from ecomove_dashboard import app

app.run(jupyter_mode='inline', jupyter_height=1400)